# Step 2: 5-Fold Stratified Cross-Validation & Multi-Model Ensembling (Optimized)
**Titanic Disaster Survival Prediction**

This notebook executes a 5-Fold Stratified Cross-Validation training pipeline across 5 diverse model architectures (FastAI Tabular NN, XGBoost, LightGBM, Random Forest, and Logistic Regression), blends out-of-fold predictions, and generates optimized test set predictions ().

## 1. Setup & Preprocessed Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import xgboost as xgb
import lightgbm as lgb
from fastai.tabular.all import *

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)

train_df = pd.read_csv('data/train_cleaned.csv')
test_df = pd.read_csv('data/test.csv')

print(f'Loaded Cleaned Train Split: {train_df.shape} | Test Set: {test_df.shape}')

## 2. Test Set Feature Engineering & WCG Signal Computation
Applying identical feature engineering and computing corrected WCG signal based strictly on train labels.

In [ ]:
title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Mlle': 'Miss', 'Mme': 'Mrs', 'Ms': 'Miss',
    'Lady': 'Noble', 'Countess': 'Noble', 'Sir': 'Noble', 'Don': 'Noble', 'Jonkheer': 'Noble',
    'Capt': 'Officer', 'Col': 'Officer', 'Major': 'Officer', 'Dr': 'Officer', 'Rev': 'Rev'
}

test_clean = test_df.copy()
test_clean['Title'] = test_clean['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
test_clean['TitleGroup'] = test_clean['Title'].map(title_mapping).fillna('Rare')
test_clean['IsWomanChild'] = ((test_clean['TitleGroup'] == 'Master') | (test_clean['Sex'] == 'female')).astype(int)
test_clean['Surname'] = test_clean['Name'].str.split(',').str[0]

test_clean['FamilySize'] = test_clean['SibSp'] + test_clean['Parch'] + 1
test_clean['IsAlone'] = (test_clean['FamilySize'] == 1).astype(int)
test_clean['FamilyType'] = pd.cut(test_clean['FamilySize'], bins=[0, 1, 4, 20], labels=['Single', 'Small', 'Large'])
test_clean['CabinDeck'] = test_clean['Cabin'].str[0].fillna('U')
test_clean['HasCabin'] = (test_clean['Cabin'].notnull()).astype(int)

# Combine train and test to get accurate ticket group counts
full_df = pd.concat([pd.read_csv('data/train.csv'), test_df], ignore_index=True)
ticket_counts = full_df['Ticket'].value_counts()
test_clean['TicketGroupSize'] = test_clean['Ticket'].map(ticket_counts)

test_clean['Fare'] = test_clean.groupby('Pclass')['Fare'].transform(lambda x: x.fillna(x.median()))
test_clean['FarePerPerson'] = test_clean['Fare'] / test_clean['TicketGroupSize']
test_clean['LogFarePerPerson'] = np.log1p(test_clean['FarePerPerson'])
test_clean['Age'] = test_clean.groupby(['TitleGroup', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))
test_clean['Age'] = test_clean['Age'].fillna(train_df['Age'].median())
test_clean['Embarked'] = test_clean['Embarked'].fillna('S')
test_clean['AgeGroup'] = pd.cut(test_clean['Age'], bins=[0, 12, 18, 50, 100], labels=['Child', 'Teen', 'Adult', 'Senior'])

# Calculate WCG Survival Rate strictly using train dataset ground truth!
train_wcg = train_df[train_df['IsWomanChild'] == 1]
ticket_wcg_train_count = train_wcg.groupby('Ticket')['PassengerId'].count()
ticket_wcg_train_survived = train_wcg.groupby('Ticket')['Survived'].sum()
surname_wcg_train_count = train_wcg.groupby(['Surname', 'Pclass'])['PassengerId'].count()
surname_wcg_train_survived = train_wcg.groupby(['Surname', 'Pclass'])['Survived'].sum()

def compute_wcg_test(row):
    ticket = row['Ticket']
    surname = row['Surname']
    pclass = row['Pclass']
    
    t_count = ticket_wcg_train_count.get(ticket, 0)
    t_surv = ticket_wcg_train_survived.get(ticket, 0)
    if t_count > 0:
        return t_surv / t_count
        
    s_count = surname_wcg_train_count.get((surname, pclass), 0)
    s_surv = surname_wcg_train_survived.get((surname, pclass), 0)
    if s_count > 0:
        return s_surv / s_count
        
    return -1.0

test_clean['WCG_Rate'] = test_clean.apply(compute_wcg_test, axis=1)

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 
            'TitleGroup', 'FamilySize', 'IsAlone', 'FamilyType', 'CabinDeck', 'HasCabin', 
            'TicketGroupSize', 'FarePerPerson', 'LogFarePerPerson', 'WCG_Rate', 'AgeGroup']

print('Test Feature Engineering Complete!')
print(test_clean[['PassengerId', 'Name', 'TitleGroup', 'FarePerPerson', 'WCG_Rate']].head(10))

## 3. 5-Fold Stratified Cross-Validation Training
Training 5-fold models across Logistic Regression, Random Forest, Extra Trees, XGBoost, and LightGBM.

In [ ]:
X = train_df[features]
y = train_df['Survived']
X_test = test_clean[features]

cat_cols = ['Sex', 'Embarked', 'TitleGroup', 'FamilyType', 'CabinDeck', 'AgeGroup']
num_cols = [c for c in features if c not in cat_cols]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=0.5),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_split=4, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=300, max_depth=5, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, eval_metric='logloss', random_state=42),
    'LightGBM': lgb.LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42, verbose=-1)
}

oof_preds = {m: np.zeros(len(train_df)) for m in models}
test_preds = {m: np.zeros(len(test_clean)) for m in models}

for name, clf in models.items():
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]
        
        pipe = SklearnPipeline([('prep', preprocessor), ('clf', clf)])
        pipe.fit(X_tr, y_tr)
        
        oof_preds[name][val_idx] = pipe.predict_proba(X_va)[:, 1]
        test_preds[name] += pipe.predict_proba(X_test)[:, 1] / 5.0
        
    acc = accuracy_score(y, (oof_preds[name] > 0.5).astype(int))
    print(f'{name} 5-Fold OOF Accuracy: {acc:.4f}')

## 4. FastAI 5-Fold Tabular Neural Network

In [ ]:
oof_preds['FastAI_NN'] = np.zeros(len(train_df))
test_preds['FastAI_NN'] = np.zeros(len(test_clean))

cat_names = ['Sex', 'Pclass', 'Embarked', 'TitleGroup', 'IsAlone', 'FamilyType', 'CabinDeck', 'AgeGroup']
cont_names = ['FarePerPerson', 'TicketGroupSize', 'WCG_Rate', 'FamilySize', 'Age', 'LogFarePerPerson']
procs = [Categorify, FillMissing, Normalize]

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y)):
    fold_df = pd.concat([train_df.iloc[train_idx], train_df.iloc[val_idx]]).reset_index(drop=True)
    splits = (list(range(len(train_idx))), list(range(len(train_idx), len(fold_df))))
    
    to = TabularPandas(fold_df, procs=procs, cat_names=cat_names, cont_names=cont_names, y_names='Survived', y_block=CategoryBlock(), splits=splits)
    dls = to.dataloaders(bs=64)
    learn = tabular_learner(dls, layers=[64, 32], metrics=accuracy)
    learn.fit_one_cycle(10, 1e-2)
    
    preds_val, _ = learn.get_preds(dl=dls.valid)
    oof_preds['FastAI_NN'][val_idx] = preds_val[:, 1].numpy()
    
    t_dl = dls.test_dl(test_clean[cat_names + cont_names])
    preds_t, _ = learn.get_preds(dl=t_dl)
    test_preds['FastAI_NN'] += preds_t[:, 1].numpy() / 5.0

nn_acc = accuracy_score(y, (oof_preds['FastAI_NN'] > 0.5).astype(int))
print(f'FastAI Neural Network 5-Fold OOF Accuracy: {nn_acc:.4f}')

## 5. Blended Ensemble Optimization

In [ ]:
oof_blend = (0.25 * oof_preds['XGBoost'] + 
             0.25 * oof_preds['LightGBM'] + 
             0.20 * oof_preds['Random Forest'] + 
             0.15 * oof_preds['FastAI_NN'] + 
             0.15 * oof_preds['Logistic Regression'])

blend_acc = accuracy_score(y, (oof_blend > 0.5).astype(int))
blend_auc = roc_auc_score(y, oof_blend)

print(f'=== 5-FOLD BLENDED ENSEMBLE OOF ACCURACY: {blend_acc:.4f} (ROC AUC: {blend_auc:.4f}) ===')

# Confusion Matrix Plot
cm = confusion_matrix(y, (oof_blend > 0.5).astype(int))
plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('5-Fold Ensemble OOF Confusion Matrix')
plt.colorbar()
plt.xticks([0, 1], ['Perished (0)', 'Survived (1)'])
plt.yticks([0, 1], ['Perished (0)', 'Survived (1)'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), horizontalalignment='center', color='white' if cm[i, j] > cm.max()/2 else 'black')

plt.tight_layout()
plt.savefig('ensemble_confusion_matrix.png', dpi=300)
plt.show()

## 6. Optimized Test Inference & Submission File Generation

In [ ]:
test_blend = (0.25 * test_preds['XGBoost'] + 
              0.25 * test_preds['LightGBM'] + 
              0.20 * test_preds['Random Forest'] + 
              0.15 * test_preds['FastAI_NN'] + 
              0.15 * test_preds['Logistic Regression'])

final_preds = (test_blend > 0.5).astype(int)

# Create submission dataframe
sub_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': final_preds
})

sub_df.to_csv('data/submission.csv', index=False)
print('Successfully saved data/submission.csv!')
print(sub_df.head(15))
print('Submission shape:', sub_df.shape)
print('Survival distribution in final test predictions:')
print(sub_df["Survived"].value_counts(normalize=True))